# Dependencies and Utilities

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from typing import Any

def save_tex(obj: Any, filename: str) -> None:
    """Exports statsmodels summaries or contrast results directly to a LaTeX file.

    Args:
        obj: The fitted regression model or the hypothesis test contrast result.
        filename: The target file name for the .tex output.
    """
    filepath = f"outputs/{filename}"

    if hasattr(obj, "summary_frame"):
        try:
            content = obj.summary_frame().to_latex(escape=False)
        except NotImplementedError:
            f_stats = pd.DataFrame({
                "F-statistic": [float(np.squeeze(obj.fvalue))],
                "p-value": [float(np.squeeze(obj.pvalue))],
                "df_num": [obj.df_num],
                "df_denom": [obj.df_denom]
            })
            content = f_stats.to_latex(index=False, escape=False)

    elif hasattr(obj, "summary"):
        content = obj.summary().as_latex()
    else:
        raise TypeError("Object type not supported for LaTeX export.")

    with open(filepath, "w") as f:
        f.write(content)

# Question 1

In [2]:
df_q1 = pd.read_excel("data/6923.Q1(26-1).xlsx", usecols=["Y", "X1", "X2", "X3", "X4"]).dropna(how="all")

mdl_1a = smf.ols("Y ~ X1 + X2 + X3 + X4", data=df_q1).fit()
save_tex(mdl_1a, "q1a_regression.tex")

for i, val in enumerate([2.00, 7.50, 0.01, 0.05], start=1):
    save_tex(mdl_1a.t_test(f"X{i} = {val}"), f"q1b_ttest_X{i}.tex")

df_q1["X4_sq"] = df_q1["X4"] ** 2
mdl_1c = smf.ols("Y ~ X4 + X4_sq", data=df_q1).fit()
save_tex(mdl_1c, "q1c_quadratic.tex")

# Question 2

In [3]:
df_q2 = pd.read_excel("data/6923.Q2(26-1).xlsx").dropna(how="all")

mdl_2 = smf.ols("V ~ ABV + AE + NECE * POLLUTE + ECE * POLLUTE", data=df_q2).fit()
save_tex(mdl_2, "q2_regression.tex")

save_tex(mdl_2.t_test("ECE + ECE:POLLUTE = 0"), "q2_ttest_ece.tex")
save_tex(mdl_2.f_test("ECE + ECE:POLLUTE = 0"), "q2_ftest_ece.tex")

# Question 3

In [4]:
df_q3 = pd.read_excel("data/6923.Q3(26-1).xlsx").dropna(
    subset=["V", "BV", "Earn", "CSR_Perf", "CSR_Info", "Assure", "Industry"]
)
fml_3 = "V ~ BV + Earn + CSR_Perf + CSR_Info + Assure"

mdl_3a = smf.ols(fml_3, data=df_q3).fit()
save_tex(mdl_3a, "q3a_ols.tex")

mdl_3b = smf.ols(fml_3, data=df_q3).fit(cov_type="cluster", cov_kwds={"groups": df_q3["Industry"]})
save_tex(mdl_3b, "q3b_clustered.tex")

ctry_cols = [c for c in df_q3.columns if c.startswith("country") and c != "country1"]
fml_3c = f"{fml_3} + " + " + ".join(ctry_cols)

mdl_3c = smf.ols(fml_3c, data=df_q3).fit(cov_type="cluster", cov_kwds={"groups": df_q3["Industry"]})
save_tex(mdl_3c, "q3c_fixed_effects.tex")

c:\Users\Zac Kienzle\Miniconda3\envs\OptionPricer\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


# Question 4

In [5]:
df_q4 = pd.read_excel("data/6923.Q4(26-1).xlsx").rename(columns=lambda x: str(x).strip()).dropna(how="all")

df_q4 = df_q4.assign(
    d_jan=(df_q4["Month"] == 1).astype(int),
    d_jul=(df_q4["Month"] == 7).astype(int),
    d_oth=(~df_q4["Month"].isin([1, 7])).astype(int),
    rp_sml_rf=df_q4["Small"] - df_q4["Riskfree"],
    rp_lrg_rf=df_q4["Large"] - df_q4["Riskfree"],
    rm_ew_rf=df_q4["EW Mkt"] - df_q4["Riskfree"],
    rm_vw_rf=df_q4["VW Mkt"] - df_q4["Riskfree"]
)

mdl_41_sml = smf.ols("rp_sml_rf ~ d_jan + rm_ew_rf", data=df_q4).fit()
save_tex(mdl_41_sml, "q41_small.tex")

mdl_41_lrg = smf.ols("rp_lrg_rf ~ d_jan + rm_ew_rf", data=df_q4).fit()
save_tex(mdl_41_lrg, "q41_large.tex")

df_sml = df_q4[["d_jan", "rm_ew_rf", "rp_sml_rf"]].rename(columns={"rp_sml_rf": "rp_rf"}).assign(d_sml=1)
df_lrg = df_q4[["d_jan", "rm_ew_rf", "rp_lrg_rf"]].rename(columns={"rp_lrg_rf": "rp_rf"}).assign(d_sml=0)
df_pool = pd.concat([df_sml, df_lrg], ignore_index=True)

mdl_41_pool = smf.ols("rp_rf ~ d_jan * d_sml + rm_ew_rf * d_sml", data=df_pool).fit()
save_tex(mdl_41_pool, "q41_pooled.tex")

mdl_42_sml = smf.ols("rp_sml_rf ~ d_oth + d_jan + d_jul + rm_vw_rf - 1", data=df_q4).fit()
save_tex(mdl_42_sml, "q42_small.tex")

mdl_42_lrg = smf.ols("rp_lrg_rf ~ d_oth + d_jan + d_jul + rm_vw_rf - 1", data=df_q4).fit()
save_tex(mdl_42_lrg, "q42_large.tex")

df_q4_3 = df_q4[df_q4["Year"] != 1987].copy()
p1 = (df_q4_3["Year"] <= 1986).astype(int)
p2 = (df_q4_3["Year"] >= 1988).astype(int)

df_q4_3 = df_q4_3.assign(
    vw_mkt=df_q4_3["VW Mkt"],
    d1_jan=p1 * df_q4_3["d_jan"],
    d1_oth=p1 * (1 - df_q4_3["d_jan"]),
    d2_jan=p2 * df_q4_3["d_jan"],
    d2_oth=p2 * (1 - df_q4_3["d_jan"])
)

mdl_43 = smf.ols("vw_mkt ~ d1_jan + d1_oth + d2_jan + d2_oth - 1", data=df_q4_3).fit()
save_tex(mdl_43, "q43_regression.tex")

save_tex(mdl_43.t_test("d1_jan = d2_jan"), "q43_ttest_jan.tex")
save_tex(mdl_43.t_test("d1_oth = d2_oth"), "q43_ttest_oth.tex")